# 🚀 Chitragupt — Gemini API Testing on Colab

Tests **Gemini Flash 3.5** (or any Gemini model) via Google AI Studio API.
Use this to experiment with Gemini's vision + reasoning capabilities
before integrating into the full Chitragupt pipeline.

### What this notebook covers

| Section | What |
|---------|------|
| 1 | Setup — install SDK + configure API key |
| 2 | Basic text generation |
| 3 | Vision — describe an image |
| 4 | Two-stage pipeline (vision → reasoning) |
| 5 | Tool calling / function calling |
| 6 | Full Chitragupt agent loop |
| 7 | Performance benchmarks |

---
## 1️⃣ Setup

In [ ]:
# @title Install Google AI Python SDK
!pip install -q google-genai
print('✅ google-genai installed')

In [ ]:
# @title Configure API key (from Colab Secrets)
from google.colab import userdata

# Set your key at 🔑 (left sidebar) → New Secret → name="GEMINI_API_KEY"
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    print('⚠️  GEMINI_API_KEY not found in Colab Secrets.')
    print('   Add it: 🔑 (left sidebar) → New Secret → name="GEMINI_API_KEY", value=your_key')
    print('   Get a key: https://aistudio.google.com/app/apikey')
else:
    print('✅ GEMINI_API_KEY loaded from Colab Secrets')

In [ ]:
# @title Import libraries
import base64, io, time, json, re
from typing import Optional
from IPython.display import display, HTML, Image as IPyImage
from google import genai
from google.genai import types

if GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)

# Available Gemini models — pick your fighter
FLASH_MODEL = 'gemini-2.5-flash-preview-05-07'  # 2.5 Flash
FLASH_THINK_MODEL = 'gemini-2.5-flash-preview-05-07'  # same model, thinking enabled via config
PRO_MODEL = 'gemini-2.5-pro-preview-05-07'      # 2.5 Pro (stronger, slower)

print('✅ Libraries imported')

---
## 2️⃣ Basic Text Generation

In [ ]:
# @title Simple text completion
if GEMINI_API_KEY:
    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents="Reply with only the word 'ready'.",
    )
    print(f'🧠 Response: {response.text}')
else:
    print('⚠️  Set GEMINI_API_KEY first')

In [ ]:
# @title Chat with history (multi-turn)
if GEMINI_API_KEY:
    chat = client.chats.create(model=FLASH_MODEL)
    r1 = chat.send_message("Say 'hello' and introduce yourself.")
    print(f'Turn 1: {r1.text}')
    r2 = chat.send_message("What was the first thing I asked you?")
    print(f'Turn 2: {r2.text}')
else:
    print('⚠️  Set GEMINI_API_KEY first')

---
## 3️⃣ Vision — Describe an Image

Upload an image from your local machine to test Gemini's vision capabilities.

In [ ]:
# @title Upload and display an image
from google.colab import files

uploaded = files.upload()

image_path = None
image_bytes = None
for fn in uploaded.keys():
    image_path = fn
    image_bytes = uploaded[fn]
    break

if image_bytes:
    display(IPyImage(data=image_bytes))
    print(f'📁 Loaded: {image_path} ({len(image_bytes)} bytes)')
else:
    print('⚠️  No image uploaded')

In [ ]:
# @title Describe the uploaded image with Gemini
if GEMINI_API_KEY and image_bytes:
    prompt = (
        "Describe everything visible in this image in detail. "
        "Include: objects, people, actions, text, colours, spatial layout, "
        "and anything that might matter for helping someone understand this scene. "
        "Be factual and specific. Do not offer advice or opinions."
    )

    t0 = time.time()
    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=[prompt, types.Part.from_bytes(data=image_bytes, mime_type='image/jpeg')],
    )
    elapsed = time.time() - t0

    print(f'⏱️  Vision took {elapsed:.1f}s')
    print('=' * 50)
    print(response.text)
else:
    print('⚠️  Upload an image and set API key first')

---
## 4️⃣ Two-Stage Pipeline (Vision → Reasoning)

Replicates the Chitragupt pipeline: describe the image with the vision model,
then feed the description into a reasoning call.

In [ ]:
# @title Stage 1: Vision description
import time

VISION_SYSTEM = (
    "Describe everything visible in this image in detail. "
    "Include: objects, people, actions, text, colours, spatial layout, "
    "and anything that might matter for helping someone understand this scene. "
    "Be factual and specific. Do not offer advice or opinions."
)

if GEMINI_API_KEY and image_bytes:
    t0 = time.time()
    vision_resp = client.models.generate_content(
        model=FLASH_MODEL,
        contents=[VISION_SYSTEM, types.Part.from_bytes(data=image_bytes, mime_type='image/jpeg')],
    )
    scene_description = vision_resp.text
    elapsed = time.time() - t0
    print(f'⏱️  Vision: {elapsed:.1f}s')
    print('📝 Description:')
    print(scene_description)
else:
    scene_description = "No image provided — using test scene."
    print('⚠️  Using placeholder description')

In [ ]:
# @title Stage 2: Reasoning with scene context
user_question = "What's happening in this scene? Give me a concise summary."  # ← Edit this!

if GEMINI_API_KEY:
    reason_prompt = (
        f"You are Chitragupt, an all-seeing assistant with access to tools.\n\n"
        f"[Camera feed]\n{scene_description}\n\n"
        f"[User]\n{user_question}\n\n"
        f"Think step by step before responding. "
        f"Be concise, practical, and helpful in your final response."
    )

    t0 = time.time()
    reason_resp = client.models.generate_content(
        model=FLASH_MODEL,
        contents=reason_prompt,
    )
    elapsed = time.time() - t0

    print(f'⏱️  Reasoning: {elapsed:.1f}s')
    print(f'⏱️  Total: {elapsed + (time.time() - t0 if False else 0):.1f}s — but separately measured above')
    print('=' * 50)
    print(reason_resp.text)
else:
    print('⚠️  Set GEMINI_API_KEY first')

---
## 5️⃣ Tool Calling / Function Calling

Gemini natively supports function calling (tools). Let's test it.

In [ ]:
# @title Define tools and test function calling
from google.genai import types

# Define a calculator tool
calculator = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="calculate",
            description="Evaluate a mathematical expression",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "expression": types.Schema(
                        type=types.Type.STRING,
                        description="The math expression to evaluate (e.g., '42 * 13')",
                    ),
                },
                required=["expression"],
            ),
        ),
        types.FunctionDeclaration(
            name="get_time",
            description="Get the current time",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "timezone": types.Schema(
                        type=types.Type.STRING,
                        description="Timezone (default UTC)",
                    ),
                },
            ),
        ),
    ],
)

# Test: ask something that requires calculation
if GEMINI_API_KEY:
    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents="What is 42 * 13?",
        config=types.GenerateContentConfig(
            tools=[calculator],
        ),
    )

    # Check if the model called a function
    if response.candidates[0].content.parts[0].function_call:
        fc = response.candidates[0].content.parts[0].function_call
        print(f'🔧 Tool called: {fc.name}({json.dumps(fc.args)})')

        # Execute the function
        if fc.name == 'calculate':
            expr = fc.args['expression']
            import math
            result = str(eval(expr, {'__builtins__': {}}, {'abs': abs, 'round': round, 'math': math}))
            print(f'📊 Result: {result}')

            # Send result back to model for final answer
            response2 = client.models.generate_content(
                model=FLASH_MODEL,
                contents=[
                    types.Content(role='user', parts=[types.Part.from_text(text='What is 42 * 13?')]),
                    types.Content(role='model', parts=[types.Part.from_function_call(name=fc.name, args=fc.args)]),
                    types.Content(role='user', parts=[types.Part.from_function_response(name=fc.name, response={'result': result})]),
                ],
                config=types.GenerateContentConfig(
                    tools=[calculator],
                    system_instruction="You are Chitragupt, a helpful assistant.",
                ),
            )
            print(f'🧠 Final answer: {response2.text}')
    else:
        print(f'🧠 Direct answer: {response.text}')
else:
    print('⚠️  Set GEMINI_API_KEY first')

---
## 6️⃣ Full Chitragupt Agent Loop

Puts it all together: vision + reasoning + tool calling in one pipeline,
matching the `ChitraguptAgent.process()` flow.

In [ ]:
# @title Full pipeline: vision → reason → tools → final answer
def tool_calculate(expression: str) -> str:
    import math
    try:
        allowed = {'abs': abs, 'round': round, 'int': int, 'float': float, 'min': min, 'max': max, 'sum': sum, 'math': math}
        return str(eval(expression, {'__builtins__': {}}, allowed))
    except Exception as e:
        return f'Error: {e}'


def tool_get_time(timezone: str = 'UTC') -> str:
    from datetime import datetime, timezone as tz
    return f'Current UTC time: {datetime.now(tz.utc).isoformat()}'


AVAILABLE_TOOLS = {
    'calculate': tool_calculate,
    'get_time': tool_get_time,
}


def chitragupt_process(user_prompt: str, image_bytes: Optional[bytes] = None) -> dict:
    """Two-stage pipeline: vision → reasoning + optional tool calls."""
    result = {'text': '', 'tool_calls': [], 'timing': {}}

    # ── Stage 1: Vision ──
    scene = None
    if image_bytes:
        t0 = time.time()
        vision_resp = client.models.generate_content(
            model=FLASH_MODEL,
            contents=[VISION_SYSTEM, types.Part.from_bytes(data=image_bytes, mime_type='image/jpeg')],
        )
        scene = vision_resp.text
        result['timing']['vision'] = round(time.time() - t0, 1)
        result['scene_description'] = scene

    # ── Stage 2: Reason ──
    reason_prompt = (
        f"You are Chitragupt, an all-seeing assistant with access to tools.\n\n"
        f"[Camera feed]\n{scene or 'No camera feed.'}\n\n"
        f"[User]\n{user_prompt}\n\n"
        f"Think step by step before responding. "
        f"If you need external information, call a tool. "
        f"Be concise, practical, and helpful in your final response."
    )

    t0 = time.time()
    reason_resp = client.models.generate_content(
        model=FLASH_MODEL,
        contents=reason_prompt,
        config=types.GenerateContentConfig(
            tools=[calculator],
        ),
    )
    result['timing']['reasoning'] = round(time.time() - t0, 1)

    full_text = reason_resp.text

    # ── Tool execution ──
    # Check for function calls in the response
    if reason_resp.candidates:
        for part in reason_resp.candidates[0].content.parts:
            if hasattr(part, 'function_call') and part.function_call is not None:
                fc = part.function_call
                tool_fn = AVAILABLE_TOOLS.get(fc.name)
                if tool_fn:
                    t1 = time.time()
                    tool_result = tool_fn(**fc.args)
                    result['tool_calls'].append({
                        'tool': fc.name,
                        'arguments': dict(fc.args),
                        'result': tool_result,
                        'time': round(time.time() - t1, 1),
                    })

    # ── Final answer (if tools were called) ──
    if result['tool_calls']:
        # Build the tool response conversation
        contents = [types.Content(role='user', parts=[types.Part.from_text(text=reason_prompt)])]
        contents.append(types.Content(role='model', parts=[types.Part.from_text(text=full_text)]))
        for tc in result['tool_calls']:
            contents.append(types.Content(
                role='user',
                parts=[types.Part.from_function_response(
                    name=tc['tool'],
                    response={'result': tc['result']},
                )],
            ))

        t0 = time.time()
        final_resp = client.models.generate_content(
            model=FLASH_MODEL,
            contents=contents,
            config=types.GenerateContentConfig(
                tools=[calculator],
            ),
        )
        result['timing']['final'] = round(time.time() - t0, 1)
        result['text'] = final_resp.text
    else:
        result['text'] = full_text

    return result


# ── Run it ──
if GEMINI_API_KEY:
    test_prompt = "What's in the image? Also, what is 15 * 27?"
    result = chitragupt_process(test_prompt, image_bytes if image_bytes else None)

    print('⏱️  Timing:', json.dumps(result['timing'], indent=2))
    print()
    if result.get('scene_description'):
        print('📝 Scene description (first 300 chars):')
        print(result['scene_description'][:300] + '...')
        print()
    if result['tool_calls']:
        print('🔧 Tool calls:')
        for tc in result['tool_calls']:
            print(f'  • {tc["tool"]}({json.dumps(tc["arguments"])}) → {tc["result"]}')
        print()
    print('🧠 Final answer:')
    print(result['text'])
else:
    print('⚠️  Set GEMINI_API_KEY first')

---
## 7️⃣ Performance Benchmarks

Compare latency across different Gemini model configurations.

In [ ]:
# @title Benchmark: vision latency
import time

MODELS_TO_TEST = [
    ('gemini-2.5-flash-preview-05-07', 'Flash 2.5'),
    ('gemini-2.5-pro-preview-05-07', 'Pro 2.5'),
]

if GEMINI_API_KEY and image_bytes:
    print(f'{"Model":<20} {"Time (s)":<10} {"Length (chars)":<15}')
    print('-' * 45)
    for model_id, label in MODELS_TO_TEST:
        t0 = time.time()
        resp = client.models.generate_content(
            model=model_id,
            contents=[VISION_SYSTEM, types.Part.from_bytes(data=image_bytes, mime_type='image/jpeg')],
        )
        elapsed = time.time() - t0
        print(f'{label:<20} {elapsed:<10.1f} {len(resp.text):<15}')
else:
    print('⚠️  Upload an image and set API key first')

In [ ]:
# @title Benchmark: reasoning latency
reasoning_tests = [
    'Say "hello"',
    'What is 42 * 13? Explain step by step.',
    'Explain quantum computing to a 10-year-old in 3 paragraphs.',
]

if GEMINI_API_KEY:
    print(f'{"Prompt":<60} {"Model":<20} {"Time (s)":<10} {"Output chars":<15}')
    print('-' * 105)
    for prompt in reasoning_tests:
        for model_id, label in MODELS_TO_TEST:
            t0 = time.time()
            resp = client.models.generate_content(
                model=model_id,
                contents=prompt,
            )
            elapsed = time.time() - t0
            display_prompt = prompt[:55] + '...' if len(prompt) > 55 else prompt
            print(f'{display_prompt:<60} {label:<20} {elapsed:<10.1f} {len(resp.text):<15}')
else:
    print('⚠️  Set GEMINI_API_KEY first')

---
## ✅ How to integrate with the Chitragupt server

Once you're happy with Gemini's performance, set these in your `server/.env`:

```
BACKEND_MODE=api
API_PROVIDER=gemini
API_MODEL=gemini-2.5-flash-preview-05-07
GEMINI_API_KEY=your_key_here
```

Then run the server:  `python -m server`

### Gemini vs. Ollama two-stage pipeline

| Aspect | Ollama (Qwen3-VL + Qwen3) | Gemini Flash 3.5 |
|--------|---------------------------|-------------------|
| Vision + reasoning | Two separate models | Single model call |
| Latency | ~10-35s total | ~2-8s total |
| Quality | Good (8B models) | Excellent |
| Cost | Free (your GPU) | Pay-per-token |
| Tool calling | Manual `<tool>` parsing | Native function calling |
| Thinking mode | Qwen3 `/think` toggle | Gemini 2.5 built-in thinking |

The existing `server/backends/gemini_backend.py` already integrates Gemini
into the Chitragupt pipeline. Use this notebook to test prompts and
configurations before deploying.